# Tutorial 04 - Time-Frequency Visualisation and Quasi-Stationarity

This notebook reconstructs longer stretches from consecutive clean segments, computes spectrograms, and applies simple stationarity metrics to identify candidate windows for later selection.

In [ ]:
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.signal import spectrogram, welch
import matplotlib.pyplot as plt
from pathlib import Path

mat_path = Path('..') / 'data' / 'MGH4J_sid001_1d8_20130718_075948.mat'
out_dir = Path('..') / 'outputs'
clean_pool_path = out_dir / 'clean_segment_pool.csv'

mat = sio.loadmat(str(mat_path), squeeze_me=True, struct_as_record=False)
bipolar = np.asarray(mat['EEG_segs_bipolar'])
channel_names = [str(x).strip() for x in np.asarray(mat['channel_names']).ravel()]
seg_start_ids = np.asarray(mat['seg_start_ids']).astype(int).ravel()
Fs = float(mat['Fs'])

if clean_pool_path.exists():
    clean_df = pd.read_csv(clean_pool_path)
    clean_segments = np.sort(clean_df['segment_index'].astype(int).to_numpy())
else:
    seg_labels = [str(x).strip() for x in np.asarray(mat['seg_masks']).ravel()]
    clean_segments = np.array([i for i, lbl in enumerate(seg_labels) if lbl == 'normal'], dtype=int)

print('Clean segments:', len(clean_segments))
print('Fs:', Fs)

## 1) Reconstruct longer stretches from consecutive clean segments

Because segments are 5 seconds each, runs of consecutive clean segment indices correspond to continuous clean stretches.

In [ ]:
runs = []
start = clean_segments[0]
prev = clean_segments[0]
for idx in clean_segments[1:]:
    if idx == prev + 1:
        prev = idx
    else:
        runs.append((start, prev))
        start, prev = idx, idx
runs.append((start, prev))

runs_df = pd.DataFrame(runs, columns=['start_seg', 'end_seg'])
runs_df['n_segments'] = runs_df['end_seg'] - runs_df['start_seg'] + 1
runs_df['duration_s'] = runs_df['n_segments'] * 5
runs_df = runs_df.sort_values('n_segments', ascending=False).reset_index(drop=True)
runs_df.head(10)

In [ ]:
chosen_run = runs_df.iloc[0]
seg_start = int(chosen_run['start_seg'])
seg_end = int(chosen_run['end_seg'])

stretch = bipolar[seg_start:seg_end + 1]  # n_seg x n_ch x 1000
stretch = np.concatenate([stretch[i] for i in range(stretch.shape[0])], axis=1)  # n_ch x time

print('Chosen run:', seg_start, 'to', seg_end)
print('Duration (s):', stretch.shape[1] / Fs)
print('Stretch shape (channels x samples):', stretch.shape)

## 2) Compute and inspect spectrogram

We inspect one representative channel first. You can repeat for other channels by changing `channel_idx`.

In [ ]:
channel_idx = 0
signal = stretch[channel_idx]

f, t, Sxx = spectrogram(
    signal,
    fs=Fs,
    window='hann',
    nperseg=400,
    noverlap=300,
    detrend='constant',
    scaling='density',
    mode='psd'
)

freq_mask = (f >= 0.5) & (f <= 40)

fig, ax = plt.subplots(figsize=(11, 4))
pcm = ax.pcolormesh(t, f[freq_mask], 10 * np.log10(Sxx[freq_mask] + 1e-20), shading='auto', cmap='viridis')
ax.set_xlabel('Time within chosen run (s)')
ax.set_ylabel('Frequency (Hz)')
ax.set_title(f'Spectrogram - channel {channel_names[channel_idx]}')
cbar = plt.colorbar(pcm, ax=ax)
cbar.set_label('Power (dB/Hz)')
plt.tight_layout()
plt.show()

## 3) Practical quasi-stationarity metrics

We define two simple metrics over 1-minute windows (step 10 s):
1. Rolling variance (`variance_median_ch`)
2. PSD self-similarity (`psd_cosine_to_prev`) between neighboring windows

Higher cosine similarity and stable variance suggest more stationary behavior.

In [ ]:
win_s = 60
step_s = 10
win_n = int(win_s * Fs)
step_n = int(step_s * Fs)

rows = []
for start in range(0, stretch.shape[1] - win_n + 1, step_n):
    end = start + win_n
    chunk = stretch[:, start:end]

    var_med = float(np.median(np.var(chunk, axis=1)))

    psd_stack = []
    for ch in range(chunk.shape[0]):
        f_w, p_w = welch(chunk[ch], fs=Fs, window='hann', nperseg=400, noverlap=200, detrend='constant')
        psd_stack.append(p_w)
    psd_stack = np.asarray(psd_stack)
    psd_med = np.median(psd_stack, axis=0)

    rows.append({
        'run_start_seg': seg_start,
        'run_end_seg': seg_end,
        'window_start_sample_in_run': start,
        'window_end_sample_in_run': end,
        'window_start_time_in_run_s': start / Fs,
        'window_end_time_in_run_s': end / Fs,
        'variance_median_ch': var_med,
        'psd_vector': psd_med
    })

station_df = pd.DataFrame(rows)

similarities = [np.nan]
for i in range(1, len(station_df)):
    a = station_df.loc[i - 1, 'psd_vector']
    b = station_df.loc[i, 'psd_vector']
    sim = float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-20))
    similarities.append(sim)
station_df['psd_cosine_to_prev'] = similarities
station_df['variance_z'] = (station_df['variance_median_ch'] - station_df['variance_median_ch'].median()) / (station_df['variance_median_ch'].mad() + 1e-12)

station_df.head()

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axs[0].plot(station_df['window_start_time_in_run_s'], station_df['variance_median_ch'], marker='o')
axs[0].set_ylabel('Median variance across channels')
axs[0].set_title('Stationarity metrics over 1-minute windows')
axs[0].spines[['top', 'right']].set_visible(False)

axs[1].plot(station_df['window_start_time_in_run_s'], station_df['psd_cosine_to_prev'], marker='o')
axs[1].set_xlabel('Window start time in run (s)')
axs[1].set_ylabel('PSD cosine similarity to previous window')
axs[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

## 4) Identify candidate stationary windows

Simple tutorial rule:
- `abs(variance_z) <= 1.5`
- `psd_cosine_to_prev >= 0.98`

These thresholds are heuristic and should be adapted after visual checks.

In [ ]:
cand_df = station_df.copy()
cand_df['is_stationary_candidate'] = (cand_df['variance_z'].abs() <= 1.5) & (cand_df['psd_cosine_to_prev'] >= 0.98)

print(cand_df['is_stationary_candidate'].value_counts())

stationary_candidates = cand_df[cand_df['is_stationary_candidate']].copy()
stationary_candidates = stationary_candidates.drop(columns=['psd_vector'])

candidate_path = out_dir / 'stationary_window_candidates.csv'
stationary_candidates.to_csv(candidate_path, index=False)
print('Saved:', candidate_path.resolve())
stationary_candidates.head()

## Summary

- We reconstructed a long clean stretch from consecutive segments.
- We visualized time-frequency behavior with a spectrogram.
- We computed practical stationarity metrics and exported candidate windows.

Tutorial 05 will select 10 one-minute segments from these candidates and export exact boundaries for later modeling.